In [1]:
import mujoco
from mujoco import mjx
import jax
import jax.numpy as jnp

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'


In [19]:
import scipy.optimize as optim

In [3]:
print("MuJoCo version:", mujoco.__version__)
print("MJX imported successfully")

MuJoCo version: 3.8.1
MJX imported successfully


In [4]:
# path to the s MuJoCo model
from pathlib import Path

path = Path('universal_robots_ur5e/scene.xml')
if path.exists():
    print("Path exists")
else:
    print("Path does not exist")

Path exists


In [5]:
# Load the MuJoCo model from an XML file and initialize the simulation state (CPU)
mj_model = mujoco.MjModel.from_xml_path(str(path))
mj_data = mujoco.MjData(mj_model)

# Transfer the MuJoCo model and data to MJX for hardware-accelerated simulation (GPU/TPU)
mjx_model = mjx.put_model(mj_model)
mjx_data = mjx.put_data(mj_model, mj_data)

In [6]:
# This function answers:If the joints are q, where is the robot hand?
# this code is copied from assignment

def forward_kinematics(q):

    new_mjx_data = mjx_data.replace(qpos=q)

    new_mjx_data = mjx.fwd_position(
        mjx_model,
        new_mjx_data
    )

    pos = new_mjx_data.site_xpos[
        mj_model.site('attachment_site').id
    ]

    mat = new_mjx_data.site_xmat[
        mj_model.site('attachment_site').id
    ]

    return pos, mat

In [7]:
## test the FK
print(forward_kinematics(jnp.zeros(6)))

(Array([-0.81700003, -0.234     ,  0.06300008], dtype=float32), Array([[ 0.9999998,  0.       , -0.       ],
       [-0.       ,  0.       , -0.9999998],
       [ 0.       ,  0.9999998,  0.       ]], dtype=float32))


In [10]:
# another test
print(forward_kinematics(jnp.array([-90,-60, 90.2,-30, 0, 0])))

(Array([-0.33843565, -0.15301135,  0.30302432], dtype=float32), Array([[-4.3914220e-01,  8.9017034e-02, -8.9399660e-01],
       [-8.7617671e-01,  1.7760700e-01,  4.4807354e-01],
       [ 1.9866624e-01,  9.8006713e-01, -5.9604645e-08]], dtype=float32))


In [12]:
# Build the error fucntion
def pose_err(q, target_pos, target_mat):

    pos, mat = forward_kinematics(q)

    pos_err = jnp.sum((pos - target_pos)**2)

    rot_err = jnp.sum((mat - target_mat)**2)

    return pos_err + rot_err


In [13]:
#jit compilation to make the run much faster
jit_err = jax.jit(pose_err)

In [14]:
jit_err_grad = jax.jit(jax.grad(pose_err, argnums=(0,)))

In [15]:
def ik_optim(target_pos, target_mat, init_guess):
    result = optim.minimize(jit_err,args=(target_pos, target_mat),x0=init_guess,
                            method='L-BFGS-B',jac=jit_err_grad)
    return result.x

In [16]:
# Write a few test cases to show the code is doing its job.
test_q = jnp.deg2rad(jnp.array([-90,-60, 90,-30, 0, 0]))
pos, rot = forward_kinematics(test_q)
print(pos, rot)

[-0.23399994  0.5519818   0.23506084] [[ 0.0000000e+00  0.0000000e+00 -9.9999964e-01]
 [-9.9999964e-01  1.1920929e-07  0.0000000e+00]
 [ 1.1920929e-07  9.9999964e-01  0.0000000e+00]]


In [25]:
## move 0.2 meters
target_pos = pos + jnp.array([0, 0, 0.3])
target_rot = rot

In [27]:
# run the optimation
target_q = ik_optim(target_pos, target_rot, init_guess=test_q)

In [28]:
res_pos, res_rot = forward_kinematics(target_q)
print(res_pos, res_rot)

[-0.23399562  0.551985    0.5350624 ] [[-6.5565109e-06 -5.0663948e-07 -9.9999988e-01]
 [-9.9999988e-01  2.0861626e-06  6.5565109e-06]
 [ 2.0563602e-06  9.9999988e-01 -5.3644180e-07]]


In [26]:
target_pos

Array([-0.23399994,  0.5519818 ,  0.5350609 ], dtype=float32)

In [29]:
res_pos

Array([-0.23399562,  0.551985  ,  0.5350624 ], dtype=float32)